# Unit II - Extracting Features, Relations from Text
## Practical Implementation for BBA Students
---

In [ ]:
# Install required libraries (run once)
!pip install nltk scikit-learn pandas matplotlib textblob wordcloud

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

## 1. Finding Implicit Features from Product Reviews

In [ ]:
from nltk import word_tokenize, pos_tag

# Sample product reviews
reviews = [
    "The camera quality is excellent and battery lasts long.",
    "It's too heavy and charges very slowly.",
    "The screen is beautiful but gets scratched easily.",
    "I love how light it is. The pictures are amazing.",
    "Battery drains quickly. The phone overheats."
]

print("=== EXPLICIT vs IMPLICIT FEATURES ===")
print("\nExplicit: 'The camera quality is excellent' -> Feature: camera quality")
print("Implicit: 'It's too heavy' -> Feature: weight (not directly mentioned!)")

# Extract adjectives and nouns to find features
print("\n=== Extracting Features using POS Tagging ===")
for review in reviews:
    tokens = word_tokenize(review)
    tagged = pos_tag(tokens)
    
    nouns = [word for word, tag in tagged if tag in ('NN', 'NNS')]
    adjectives = [word for word, tag in tagged if tag in ('JJ', 'JJS', 'JJR')]
    
    print(f"\nReview: {review}")
    print(f"  Features (Nouns): {nouns}")
    print(f"  Opinions (Adjectives): {adjectives}")

## 2. Finding Opinion Phrases and Their Polarity

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# Opinion phrases and their polarity
opinion_phrases = [
    "excellent quality",
    "terrible service",
    "not bad at all",
    "could be better",
    "very good product",
    "extremely poor",
    "fairly good",
    "absolutely amazing"
]

print("=== OPINION PHRASES AND POLARITY ===")
print(f"{'Phrase':<25} {'Compound Score':<15} {'Polarity'}")
print("-" * 55)

for phrase in opinion_phrases:
    scores = sia.polarity_scores(phrase)
    compound = scores['compound']
    
    if compound >= 0.05:
        polarity = "POSITIVE ✓"
    elif compound <= -0.05:
        polarity = "NEGATIVE ✗"
    else:
        polarity = "NEUTRAL ~"
    
    print(f"{phrase:<25} {compound:<15.3f} {polarity}")

## 3. Context-Specific Word Semantic Orientation

In [ ]:
# Same word, different contexts -> different sentiment
context_examples = {
    "cheap": {
        "price_context": "This phone is cheap and affordable.",     # POSITIVE
        "quality_context": "The material feels cheap and flimsy."    # NEGATIVE
    },
    "small": {
        "phone_context": "The phone is small and portable.",         # POSITIVE
        "screen_context": "The screen is too small to read."         # NEGATIVE
    },
    "long": {
        "battery_context": "Battery lasts long, very happy.",        # POSITIVE
        "loading_context": "Loading time is too long."               # NEGATIVE
    }
}

print("=== CONTEXT-SPECIFIC WORD ORIENTATION ===")
print("The SAME word can be positive or negative depending on context!\n")

for word, contexts in context_examples.items():
    print(f"Word: '{word}'")
    for ctx_name, sentence in contexts.items():
        score = sia.polarity_scores(sentence)['compound']
        sentiment = "POSITIVE" if score > 0 else "NEGATIVE"
        print(f"  {ctx_name}: \"{sentence}\" -> {sentiment} ({score:.2f})")
    print()

## 4. TF-IDF Analysis

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Sample documents
documents = [
    "The phone has great camera and battery life.",
    "Battery drains fast but camera quality is good.",
    "Screen display is bright and colorful.",
    "The phone camera takes amazing photos in low light.",
    "Screen is too small for watching videos."
]

# Calculate TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(documents)

# Create a readable DataFrame
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(tfidf_matrix.toarray(), 
                   columns=feature_names,
                   index=[f'Doc {i+1}' for i in range(len(documents))])

print("=== TF-IDF MATRIX ===")
print("Higher score = word is MORE important in that specific document\n")
print(df.round(3).to_string())

# Show top words per document
print("\n=== TOP 3 IMPORTANT WORDS PER DOCUMENT ===")
for i, doc in enumerate(documents):
    row = df.iloc[i]
    top_words = row.nlargest(3)
    print(f"\nDoc {i+1}: \"{doc[:50]}...\"")
    for word, score in top_words.items():
        print(f"  {word}: {score:.3f}")

## 5. Zipf's Law Visualization

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

# Combine all documents
all_text = " ".join(documents).lower()
words = [w for w in word_tokenize(all_text) if w.isalpha()]

# Count word frequencies
word_freq = Counter(words)
sorted_freq = sorted(word_freq.values(), reverse=True)

# Add more sample text for better visualization
sample_text = """The phone has excellent camera quality and the battery life is amazing.
I love the screen display and the camera takes great photos. The battery charges fast.
The phone is lightweight and the camera is the best feature. Screen resolution is high.
Battery performance is good but the phone heats up. Camera zoom is excellent.
The display is bright and colorful. Phone camera is better than expected."""

words_large = [w for w in word_tokenize(sample_text.lower()) if w.isalpha()]
freq_large = Counter(words_large)
sorted_freq_large = sorted(freq_large.values(), reverse=True)

# Plot Zipf's Law
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regular plot
ranks = range(1, len(sorted_freq_large) + 1)
axes[0].bar(ranks, sorted_freq_large, color='steelblue')
axes[0].set_xlabel('Word Rank', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title("Zipf's Law: Word Frequency vs Rank", fontsize=14)

# Log-log plot (should be roughly linear)
axes[1].scatter(np.log10(list(ranks)), np.log10(sorted_freq_large), color='coral', s=50)
axes[1].set_xlabel('log(Rank)', fontsize=12)
axes[1].set_ylabel('log(Frequency)', fontsize=12)
axes[1].set_title("Zipf's Law: Log-Log Plot (should be ~ straight line)", fontsize=14)

plt.tight_layout()
plt.savefig('zipfs_law.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nTop 10 most frequent words:")
for word, freq in freq_large.most_common(10):
    print(f"  {word}: {freq} times")

## 6. Bind TF-IDF Function (sklearn)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Training corpus
train_docs = [
    "phone camera battery screen",
    "laptop keyboard display processor",
    "tablet screen touch stylus"
]

# New document to transform using the SAME IDF values
new_docs = [
    "camera screen battery",
    "keyboard processor"
]

# Fit on training data (learn IDF values)
vectorizer = TfidfVectorizer()
train_tfidf = vectorizer.fit_transform(train_docs)

# Transform new data using BOUND IDF values
new_tfidf = vectorizer.transform(new_docs)

print("=== BIND TF-IDF ===")
print("Key Concept: IDF values are learned from training data")
print("New documents use the SAME IDF values (bound/fixed)\n")

print("Training vocabulary:", vectorizer.get_feature_names_out())
print("\nIDF Values (learned from training):")
for word, idf in zip(vectorizer.get_feature_names_out(), vectorizer.idf_):
    print(f"  {word}: {idf:.3f}")

# Find most similar training doc for each new doc
print("\n=== Document Similarity ===")
sim_matrix = cosine_similarity(new_tfidf, train_tfidf)
for i, new_doc in enumerate(new_docs):
    most_similar = sim_matrix[i].argmax()
    print(f"New doc: '{new_doc}' -> Most similar to: '{train_docs[most_similar]}' (score: {sim_matrix[i][most_similar]:.3f})")

## 7. Relation Extraction (Simple Pattern-Based)

In [ ]:
import re

# Sample sentences with relations
sentences = [
    "Microsoft acquired GitHub for 7.5 billion dollars.",
    "Elon Musk is the CEO of Tesla.",
    "Apple was founded in Cupertino, California.",
    "Google acquired YouTube in 2006.",
    "Satya Nadella works at Microsoft."
]

# Define relation patterns
patterns = {
    "ACQUISITION": r"(\w+)\s+acquired\s+(\w+)",
    "CEO_OF": r"(\w+\s\w+)\s+is the CEO of\s+(\w+)",
    "FOUNDED_IN": r"(\w+)\s+was founded in\s+([\w\s,]+)",
    "WORKS_AT": r"(\w+\s\w+)\s+works at\s+(\w+)"
}

print("=== RELATION EXTRACTION ===")
print("Extracting structured relationships from text\n")

for sentence in sentences:
    print(f"Text: \"{sentence}\"")
    for rel_type, pattern in patterns.items():
        match = re.search(pattern, sentence)
        if match:
            entity1 = match.group(1)
            entity2 = match.group(2)
            print(f"  -> Relation: {entity1} --[{rel_type}]--> {entity2}")
    print()

## 8. Subsequence Kernel (String Similarity)

In [ ]:
def subsequence_kernel(s, t, k=2, decay=0.5):
    """
    Simple Subsequence Kernel (SSK) for measuring string similarity.
    k: length of subsequences to consider
    decay: penalty for gaps (0 to 1)
    """
    # Count common subsequences of length k
    common = 0
    s_words = s.lower().split()
    t_words = t.lower().split()
    
    # Find common word subsequences
    for i in range(len(s_words) - k + 1):
        subseq_s = tuple(s_words[i:i+k])
        for j in range(len(t_words) - k + 1):
            subseq_t = tuple(t_words[j:j+k])
            if subseq_s == subseq_t:
                common += decay ** (abs(i - j))  # Penalize by position difference
    return common

# Test string kernel on relation patterns
sentences_pairs = [
    ("Microsoft acquired GitHub", "Google acquired YouTube"),
    ("Microsoft acquired GitHub", "Apple released iPhone"),
    ("CEO of Microsoft", "CEO of Google"),
    ("CEO of Microsoft", "Founded in California"),
]

print("=== SUBSEQUENCE KERNEL FOR RELATION EXTRACTION ===")
print("Higher score = sentences share more structural patterns\n")

for s1, s2 in sentences_pairs:
    score = subsequence_kernel(s1, s2)
    print(f"  '{s1}' vs '{s2}'")
    print(f"  -> Kernel score: {score:.3f}  {'(Similar pattern!)' if score > 0 else '(Different pattern)'}\n")